# 04 Statistical Analysis

**Project:** F1 Driver vs Strategy

**Author:** Ayema Qureshi

**Stage:** Silver layer (feature-enriched tables in, statistical findings out)

## Purpose
This notebook runs formal statistical tests on the patterns surfaced in the EDA (notebook 03) to answer the research question: does winning in Formula 1 come down to the driver, the team strategy, or both, and has that balance shifted across four regulatory eras? The charts gave visual impressions. This notebook checks whether those impressions are real or just noise, and puts a number and a confidence level on each one.

## Inputs
Feature tables from `data/processed/`: results, team_season, driver_season, avg_pit_stop. The `era` label is added to team_season at load time, since that table is built at constructor-season grain and does not carry era.

## Outputs
No files written. This notebook produces statistical findings and interpretations documented inline. It reads from `data/processed/` and never writes back to it.

## Statistical Testing Steps
1. Load the four feature tables from `data/processed/` and add era to team_season via pd.cut
2. Build the clean analytical frame (finished races from a real grid slot): drops DNFs and pit lane starts, matches the 5,337-row EDA frame
3. Test 1: grid vs finish Pearson correlation per era, with 95% confidence intervals and a Spearman robustness check
4. Test 2: position_change across eras, Kruskal-Wallis omnibus then pairwise Welch t-tests with Bonferroni correction and Mann-Whitney backup
5. Test 3: teammate_gap across eras, same omnibus-then-pairwise structure with Mann-Whitney as the primary test, plus a bootstrap on the 2021 median
6. Test 4: pit stop duration vs constructor points, Pearson and Spearman with a 95% confidence interval
7. Sanity checks: test whether the confounds named in Tests 2, 3, and 4 (DNF rate, season length) actually change the conclusions

## The Four Tests at a Glance

**1. Grid vs Finish Correlation** (baseline, race grain)
Does starting position predict finishing position? Pearson r stays strong across all four eras (0.78, 0.76, 0.75, 0.73), with narrow confidence intervals and Spearman agreeing. A slight, steady softening in the recent era. Points to the car doing early work, since grid is set by qualifying.

**2. Position Change Across Eras** (driver skill, race grain)
Did position movement change over time? Medians are flat but means drop from the early eras to the recent ones. The difference is real (omnibus significant) but partly confounded by reliability: more cars retired in the early eras, so finishers gained places without overtaking.

**3. Teammate Gap Across Eras** (driver skill, strongest proxy, team-season grain)
Did the gap between teammates shift by era? The medians look different (15, 27.5, 9, 22) but the omnibus is not significant (p = 0.34) and no pairwise comparison survives correction. The within-era spread is too large to see through, and 2021 (n = 10) has a bootstrap median range of roughly 6 to 88. The gap is large in every era but does not measurably differ across them.

**4. Pit Stop Speed vs Constructor Points** (team strategy, team-season grain)
Do faster pit crews score more points? A moderate negative correlation (Pearson -0.46, Spearman -0.52): slower pits, fewer points. Real but weaker than the grid-finish link, so strategy matters but is not the main lever.

## Key Decisions
1. **Read-only notebook.** This notebook reads from `data/processed/` and never writes back. Permanent features belong upstream in notebook 02. Era is added to team_season in memory only, for grouping.
2. **Omnibus before pairwise.** Every multi-era test checks whether any difference exists at all before comparing individual pairs, to avoid cherry-picking significant comparisons.
3. **Correction for multiple comparisons.** Bonferroni applied whenever six era pairs are tested, to guard against false positives.
4. **Right test for the shape of the data.** Median over mean and Mann-Whitney over t-test wherever the data is skewed (teammate gap especially). Spearman run alongside Pearson as a rank-based robustness check.
5. **Small samples handled honestly.** 2021 is one season (n = 10). A bootstrap shows exactly how untrustworthy its statistics are, rather than reporting them at face value.
6. **A non-significant result means "no difference detected," not "no difference exists."** Especially where power is low, as in 2021.
7. **Confounds are tested, not just named.** The DNF and season-length limitations are checked directly to see whether they change any conclusion.

In [1]:
from scipy import stats
import numpy as np
import pandas as pd

In [2]:
# Load feature-enriched tables written by notebook 02 (read-only, never rewritten here)
results       = pd.read_csv('../data/processed/results.csv')
team_season   = pd.read_csv('../data/processed/team_season.csv')
driver_season = pd.read_csv('../data/processed/driver_season.csv')
avg_pit_stop  = pd.read_csv('../data/processed/avg_pit_stop.csv')

In [3]:
# team_season is built at constructor-season grain and does not carry an era label.
# Add it here so era-based tests can group on it.
team_season['era'] = pd.cut(
    team_season['year'],
    bins=[2009, 2013, 2020, 2021, 2024],
    labels=['2010-2013', '2014-2020', '2021', '2022-2024']
)

### Test 1: Does starting position predict finishing position?

**What I did**
- Measured the correlation between where a driver started (grid) and where they finished (position)
- Ran it separately for each of the four eras, so I could see if the relationship shifted over time
- Used Pearson correlation as the main measure (it captures a straight-line relationship, which is what grid vs finish is)
- Added a 95% confidence interval to each result, so I know how sure I am of the number
- Ran a Spearman correlation alongside it as a sanity check (Spearman only cares about rank order, so if it agrees with Pearson, the relationship is real and not an artifact)
- Ran this on finished races only: dropped DNFs (null position) and pit lane starts (grid = 0, which is a code, not a real grid slot)

**What I found**
- The correlation was strong and steady across all four eras: 0.78, 0.76, 0.75, 0.73
- The confidence intervals were narrow (each spanned only about 0.05 to 0.09), so these numbers are reliable, not lucky
- Spearman matched Pearson almost exactly in every era (0.78, 0.77, 0.75, 0.74), which confirms the relationship holds up
- All p-values were basically zero, so none of this is down to chance

**What it means in plain English**
- If you tell me where a driver started, I can predict fairly accurately where they will finish
- Starting near the front usually means finishing near the front, starting at the back usually means finishing at the back
- Starting position is decided by qualifying, and qualifying is mostly about the car's pace
- So this points to the car doing a lot of the work before the race even begins
- There is a small but steady drop across the eras (0.78 down to 0.73)
- That means starting position still predicts the finish strongly today, but slightly less tightly than it did in the early 2010s
- In plain terms: modern racing leaves a little more room to move on Sunday

**Caveats (limitations to keep in mind)**
- A strong correlation here does not prove the car is everything, it only shows that starting position matters a lot
- Correlation is about predictability, not about how much drivers move, that movement question is a different chart
- This test cannot tell me why the grip loosened slightly in recent eras (could be rule changes, closer cars, or track characteristics), it only shows that it did
- The driver's own contribution shows up more clearly in the teammate gap test, where both drivers have the same car

In [4]:
# Clean analytical frame for correlation: finished races from a real grid slot
# Drop DNFs (position is null) and pit lane starts (grid == 0, a code not a real slot)
results_clean = results[(results['position'].notna()) & (results['grid'] != 0)].copy()
# quick check that the row count matches the EDA frame
print(results_clean.shape)

(5337, 11)


In [5]:
# Test 1: grid vs finish correlation, per era
# Pearson r = strength of the straight-line link between where you start and finish
# Higher r means starting position locks in your result more tightly

eras = ['2010-2013', '2014-2020', '2021', '2022-2024']

for era in eras:
    era_rows = results_clean[results_clean['era'] == era]

    grid = era_rows['grid']
    finish = era_rows['position']

    # pearson r and its p-value
    r, p = stats.pearsonr(grid, finish)

    print(era)
    print('  rows:', len(era_rows))
    print('  pearson r:', round(r, 3))
    print('  p-value:', p)
    print()

2010-2013
  rows: 1492
  pearson r: 0.775
  p-value: 5.25270027959493e-299

2014-2020
  rows: 2307
  pearson r: 0.756
  p-value: 0.0

2021
  rows: 378
  pearson r: 0.75
  p-value: 1.3450256195256444e-69

2022-2024
  rows: 1160
  pearson r: 0.727
  p-value: 3.0917462167089846e-191



In [6]:
def pearson_ci(r, n):
    # Fisher transformation to get a 95% confidence interval around r
    z = np.arctanh(r)
    se = 1 / np.sqrt(n - 3)
    lo = np.tanh(z - 1.96 * se)
    hi = np.tanh(z + 1.96 * se)
    return lo, hi

for era in eras:
    era_rows = results_clean[results_clean['era'] == era]
    grid = era_rows['grid']
    finish = era_rows['position']

    r, p = stats.pearsonr(grid, finish)
    lo, hi = pearson_ci(r, len(era_rows))
    rho, rho_p = stats.spearmanr(grid, finish)

    print(era)
    print('  pearson r:', round(r, 3), '  95% CI:', (round(lo, 3), round(hi, 3)))
    print('  spearman:', round(rho, 3))
    print()


2010-2013
  pearson r: 0.775   95% CI: (0.754, 0.794)
  spearman: 0.778

2014-2020
  pearson r: 0.756   95% CI: (0.738, 0.773)
  spearman: 0.765

2021
  pearson r: 0.75   95% CI: (0.703, 0.791)
  spearman: 0.753

2022-2024
  pearson r: 0.727   95% CI: (0.699, 0.753)
  spearman: 0.735



### Test 2: Did position movement change across eras?

**The question**
- Does the amount drivers move from start to finish (position_change) differ from one era to another?

**Why I chose these tests**
- I ran a Kruskal-Wallis omnibus test first because it answers one honest question up front: is there ANY difference across the four eras at all? If the answer were no, there would be no point digging into pairs. Checking pairs first and only reporting the ones that look interesting is a rookie mistake, so the omnibus is the gate.
- I used a Welch t-test for the pairwise comparisons because it compares two averages without assuming the two eras have the same spread. My eras have different sizes and different spreads, so Welch is the safer choice than a standard t-test.
- I ran a Mann-Whitney test next to each t-test as a sanity check. It does not care about averages, only about rank order, so if it agrees with the t-test I can trust the result is not an artifact of skew or outliers.
- I applied a Bonferroni correction (multiplied each p-value by 6) because I ran six comparisons, and every extra test raises the chance of a false alarm. Multiplying by six holds the bar higher so I do not get fooled by luck.

**What I did**
- Measured position_change (grid minus finish, positive means places gained) for each era
- Ran the omnibus, then all six pairwise comparisons, on the same clean frame as Test 1 (finished races, real grid slots)

**What I found**
- The medians were basically flat across eras (1.0, 1.0, 0.5, 1.0), matching Chart 2
- But the means were not flat: about 1.57 and 1.36 in the two early eras, dropping to about 0.83 in both recent eras
- The omnibus confirmed a real difference exists somewhere (p basically 0)
- The pairwise tests located it: the two early eras are not different from each other, the two recent eras are not different from each other, but early-vs-recent IS a real difference
- Plain version: finishers gained more places on average in the early 2010s than they do now

**What it means in plain English**
- The typical driver (the median) moved about the same in every era, roughly one position
- But the average movement shrank over time, because the big place-gainers became less common in recent years
- On its own that looks like "there was more overtaking in the early 2010s," but that reading is not safe (see caveats)

**Caveats (limitations to keep in mind)**
- This difference is confounded by reliability. My DNF check found retirements dropped from about 17% (2010-2020) to about 11% (2021 onward)
- When more cars ahead retire, finishers gain places without overtaking anyone, so some of the early-era movement is cars breaking down, not drivers charging through the field
- So this test cannot claim there was more real overtaking in the early 2010s, only that finishers gained more net places, for reasons that include reliability
- The mean and median disagreeing is the lesson itself: the average was pulled by the big movers in the tail, while the typical driver barely changed
- Position movement is still not where the clearest driver-skill signal lives, that is the teammate gap test

In [7]:
from itertools import combinations

# Test 2: does position movement (position_change) differ across eras?
# position_change = grid - finish. Positive means the driver gained places.
# Chart 2 showed the medians looked flat, this test checks the averages properly.

# same clean frame as Test 1, position_change has no nulls here
results_clean['position_change'] = results_clean['grid'] - results_clean['position']

# first, describe each era so I can see means vs medians side by side
for era in eras:
    pc = results_clean[results_clean['era'] == era]['position_change']
    print(f"{era}: mean={round(pc.mean(), 2)}, median={pc.median()}, n={len(pc)}")
print()

# omnibus check first: does ANY era differ at all? (Kruskal-Wallis, non-parametric)
groups = [results_clean[results_clean['era'] == era]['position_change'] for era in eras]
h_stat, p_omnibus = stats.kruskal(*groups)
print(f"Kruskal-Wallis omnibus p-value: {round(p_omnibus, 5)}")
print()

# then pairwise: compare every era against every other (6 pairs)
# Welch t-test (does not assume equal variance), plus Mann-Whitney as a sanity check
# 6 tests means more chances for a false positive, so I multiply each p by 6 (Bonferroni)
pairs = list(combinations(eras, 2))

for a, b in pairs:
    xa = results_clean[results_clean['era'] == a]['position_change']
    xb = results_clean[results_clean['era'] == b]['position_change']

    t, p = stats.ttest_ind(xa, xb, equal_var=False)
    u, p_mw = stats.mannwhitneyu(xa, xb)

    p_corrected = min(p * 6, 1.0)  # Bonferroni: strict, guards against false positives
    verdict = 'DIFFERENT' if p_corrected < 0.05 else 'no real difference'

    print(f"{a} vs {b}")
    print(f"  mean diff: {round(xa.mean() - xb.mean(), 2)}")
    print(f"  t-test p (corrected): {round(p_corrected, 4)} -> {verdict}")
    print(f"  mann-whitney p: {round(p_mw, 4)}")
    print()

2010-2013: mean=1.57, median=1.0, n=1492
2014-2020: mean=1.36, median=1.0, n=2307
2021: mean=0.83, median=0.5, n=378
2022-2024: mean=0.84, median=1.0, n=1160

Kruskal-Wallis omnibus p-value: 0.0

2010-2013 vs 2014-2020
  mean diff: 0.2
  t-test p (corrected): 0.8861 -> no real difference
  mann-whitney p: 0.006

2010-2013 vs 2021
  mean diff: 0.74
  t-test p (corrected): 0.0078 -> DIFFERENT
  mann-whitney p: 0.0002

2010-2013 vs 2022-2024
  mean diff: 0.73
  t-test p (corrected): 0.0 -> DIFFERENT
  mann-whitney p: 0.0

2014-2020 vs 2021
  mean diff: 0.54
  t-test p (corrected): 0.0785 -> no real difference
  mann-whitney p: 0.0329

2014-2020 vs 2022-2024
  mean diff: 0.53
  t-test p (corrected): 0.0014 -> DIFFERENT
  mann-whitney p: 0.0006

2021 vs 2022-2024
  mean diff: -0.01
  t-test p (corrected): 1.0 -> no real difference
  mann-whitney p: 0.9003



### Test 3: Does the teammate gap differ across eras?

**The question**
- The teammate gap is the points difference between two drivers in the same car in a season
- Same car, different results, so a big gap points to the driver, not the machine
- This is my sharpest driver-skill proxy, so this is the most important test in the notebook

**Why I chose these tests**
- I ran a Kruskal-Wallis omnibus first, same as Test 2, to check if ANY era differs before looking at pairs
- I made Mann-Whitney the primary pairwise test here, not the t-test, because the teammate gap is badly skewed: most teams have a small gap, a few have a huge one. A rank-based test is not thrown off by those big outliers, while a test of averages would be
- I applied a Bonferroni correction (times 6) for the six comparisons
- For 2021 specifically, I ran a bootstrap to measure how trustworthy its median is, because 2021 has only 10 team-seasons

**What I did**
- Compared teammate_gap across the four eras using the team_season table
- Reported the median as the headline number, because the mean is dragged around by outliers here
- Bootstrapped the 2021 median: resampled its 10 teams 10,000 times to see how much the median moves, with a fixed random seed so the range reproduces exactly on every run

**What I found**
- The medians looked different at first glance: 15, 27.5, 9, 22 (Mercedes era highest, 2021 lowest), same as Chart 3
- But the omnibus p-value was 0.34, well above 0.05, so I fail to reject the null: there is no difference across eras that I can statistically confirm
- Every pairwise comparison also came back not significant
- The reason is the spread: within every era the gaps range from near zero to nearly 200, so the differences between era medians get drowned out by the noise inside each era
- The 2021 median of 9 has a bootstrap 95% range of roughly 6 to 88, meaning that number is not stable enough to trust on its own

**What it means in plain English**
- The teammate gap is large in every era, which does support the idea that drivers matter a lot (same car, big points differences)
- But I cannot show that the gap changed across eras. The "Mercedes era had the biggest gap" and "2021 was the closest" stories are visible on the chart but do not survive a significance test
- 2021 in particular is too small and too scattered to say anything confident about
- Combined with Tests 1 and 2, the picture is consistent: the driver-versus-car balance looks remarkably stable across 15 years

**Caveats (limitations to keep in mind)**
- A non-significant result does not prove the eras are identical, it means I could not detect a difference given this much within-era noise
- 2021 has only 10 team-seasons, which gives the test very little power, so a real difference could exist and still be invisible here
- The teammate gap is raw season points, and recent seasons have more races, so recent gaps are inflated a little just by there being more races to accumulate points over
- The mean is not a safe summary here because a few huge gaps (up to 270) drag it upward, which is why I led with the median

In [8]:
# Test 3: does the teammate gap differ across eras?
# teammate_gap = points difference between the two teammates in the same car, per season.
# Same car, different results, so a big gap points to the driver.
# This is the sharpest driver-skill proxy, so it's the most important test.

# describe each era: mean, median, std, n. The gap is very skewed, so median matters most.
for era in eras:
    gap = team_season[team_season['era'] == era]['teammate_gap']
    print(era, ' median:', gap.median(), ' mean:', round(gap.mean(), 1),
          ' std:', round(gap.std(), 1), ' n:', len(gap))
print()

# omnibus first: is there ANY difference across the four eras?
groups = [team_season[team_season['era'] == era]['teammate_gap'] for era in eras]
h_stat, p_omnibus = stats.kruskal(*groups)
print('Kruskal-Wallis omnibus p-value:', round(p_omnibus, 4))
print()

# pairwise: Mann-Whitney is the PRIMARY test here (the gap is skewed with big outliers,
# so a rank-based test is safer than comparing averages). t-test shown alongside for contrast.
pairs = list(combinations(eras, 2))
for a, b in pairs:
    ga = team_season[team_season['era'] == a]['teammate_gap']
    gb = team_season[team_season['era'] == b]['teammate_gap']

    u, p_mw = stats.mannwhitneyu(ga, gb)
    p_mw_corrected = min(p_mw * 6, 1.0)  # Bonferroni for 6 comparisons

    print(a, 'vs', b, ' Mann-Whitney p (corrected):', round(p_mw_corrected, 3))
print()

# 2021 trust check: 2021 has only 10 team-seasons, so how stable is its median of 9?
# Bootstrap: resample those 10 teams 10,000 times, take the median each time,
# and see how wide the range of medians is. A wide range means the 9 is not trustworthy.
gap_2021 = team_season[team_season['era'] == '2021']['teammate_gap']
print('2021 values sorted:', sorted(gap_2021.round(1).tolist()))

np.random.seed(42)
boot_medians = []
for i in range(10000):
    sample = np.random.choice(gap_2021, size=len(gap_2021), replace=True)
    boot_medians.append(np.median(sample))

low = round(np.percentile(boot_medians, 2.5), 1)
high = round(np.percentile(boot_medians, 97.5), 1)
print('2021 median:', gap_2021.median(), ' but 95% range from bootstrap:', (low, high))

2010-2013  median: 15.0  mean: 37.0  std: 50.8  n: 47
2014-2020  median: 27.5  mean: 37.6  std: 41.2  n: 72
2021  median: 9.0  mean: 52.6  std: 73.0  n: 10
2022-2024  median: 22.0  mean: 52.3  std: 69.3  n: 30

Kruskal-Wallis omnibus p-value: 0.3449

2010-2013 vs 2014-2020  Mann-Whitney p (corrected): 0.85
2010-2013 vs 2021  Mann-Whitney p (corrected): 1.0
2010-2013 vs 2022-2024  Mann-Whitney p (corrected): 0.622
2014-2020 vs 2021  Mann-Whitney p (corrected): 1.0
2014-2020 vs 2022-2024  Mann-Whitney p (corrected): 1.0
2021 vs 2022-2024  Mann-Whitney p (corrected): 1.0

2021 values sorted: [0.0, 4.5, 7.0, 7.0, 9.0, 9.0, 46.0, 78.0, 166.5, 198.5]
2021 median: 9.0  but 95% range from bootstrap: (5.8, 87.8)


### Test 4: Does pit stop speed relate to constructor points?

**The question**
- Do teams with faster pit crews score more points across a season?
- This is the team-strategy side of my research question, the counterweight to the driver proxies

**Why I chose these tests**
- I used Pearson correlation for the main measure, to capture the straight-line link between pit speed and points
- I ran Spearman alongside it as a robustness check, because Chart 5 looked slightly curved. Spearman only asks "as one goes up, does the other move steadily the other way," so it is not fooled by a non-straight shape
- I added a 95% confidence interval so I know how firm the number is

**What I did**
- Built constructor_points (both drivers' grand prix points added together) per team-season
- Merged in each team's average pit stop duration, giving 147 team-seasons (the 2010 teams drop out because pit data starts in 2011)
- Correlated pit stop duration against constructor points

**What I found**
- Pearson r = -0.46, 95% CI -0.58 to -0.32, and Spearman rho = -0.52, both with p-values near zero
- The correlation is negative: as pit stop duration goes up (slower crew), constructor points go down
- In plain terms, faster pit stops go with more points, slower pit stops go with fewer
- The relationship is moderate, not strong (weaker than the 0.75 grid-to-finish link), which fits the idea that pit speed is one part of team execution, not the whole story
- Concretely: no team averaging slower than about 26 seconds ever cleared 155 points in a season, while faster teams reached as high as 790

**What it means in plain English**
- Pit crew speed genuinely tracks with team success, so the strategy and execution side matters
- But it matters less than starting position does, so it is a real lever, not the main one
- For the stakeholder: investing in fast, reliable pit execution is worth it, but it will not by itself turn a slow team into a winning one

**Caveats (limitations to keep in mind)**
- This is correlation, not cause. Fast pit stops might not create points directly. Strong teams tend to have both good cars and well-drilled crews, so some of this link is the two traveling together
- Pit stop duration does not account for pit lane length, which varies by circuit, so some of the spread is track differences, not crew skill
- Constructor points are raw season totals, so more recent seasons with more races can reach higher totals
- This says nothing about pit stop timing (which lap the team pits on), only pit stop speed. Timing strategy is a separate lever I did not test

In [9]:
# Test 4: does pit stop speed relate to how many points a team scores?
# This is the strategy side of the research question (Chart 5).
# avg_pit_stop_duration = how fast the crew works. constructor_points = both drivers' GP points.

# build the frame: add both drivers' points together, then attach each team's avg pit stop
team_season['constructor_points'] = team_season['points_high'] + team_season['points_low']
merged = team_season.merge(avg_pit_stop, on=['constructorId', 'year'], how='inner')
print('team-seasons with pit data:', len(merged))  # 147, the 2010 teams drop out (no pit data)

pit = merged['avg_pit_stop_duration']
points = merged['constructor_points']

# Pearson for the straight-line link, Spearman as the robustness check
# (Chart 5 looked a bit curved, so a rank-based test that only needs "one goes up as the other goes down" is safer)
r, p_pearson = stats.pearsonr(pit, points)
rho, p_spearman = stats.spearmanr(pit, points)

# 95% CI on the Pearson r, same Fisher method as Test 1
lo, hi = pearson_ci(r, len(merged))

print('Pearson r:', round(r, 3), ' 95% CI:', (round(lo, 3), round(hi, 3)), ' p:', p_pearson)
print('Spearman rho:', round(rho, 3), ' p:', p_spearman)

team-seasons with pit data: 147
Pearson r: -0.462  95% CI: (-0.58, -0.324)  p: 3.958655642922133e-09
Spearman rho: -0.523  p: 1.0952231376958453e-11


## Sanity Checks on the Limitations I Found

After writing up Tests 2, 3, and 4, I listed several limitations I was worried about. Rather than just leaving them as warnings, I asked Claude to help me come up with specific checks to actually test whether those limitations changed my conclusions. Claude proposed the three checks below and wrote the code. I ran them, verified the numbers, and wrote the interpretations.

### Check A: Is the Test 2 movement difference just caused by retirements?
**The worry**
- In Test 2, finishers gained more places in the early eras. I suspected part of that was cars ahead retiring (more DNFs early), not real overtaking.

**What I did**
- Counted DNFs per race, then re-ran the early-vs-recent comparison using only "clean" races (below-median DNF counts), where retirements can't do much of the work.

**What I found**
- On clean races, the early-era movement dropped a little (2010-2013 fell from 1.57 to 1.39), so DNFs DO inflate it
- But the early-vs-recent difference stayed significant (p = 0.001)
- Conclusion: reliability explains part of the early-era movement, mostly in 2010-2013, but not all of it. There is still genuinely more net movement in the early eras beyond dead cars.

### Check B: Does the teammate gap result depend on season length?
**The worry**
- Recent seasons have more races, so raw total-points gaps could be inflated just by there being more races.

**What I did**
- Rebuilt the teammate gap using points-per-race instead of total points, then re-ran the test across eras.

**What I found**
- The result stayed not significant (p went from 0.34 to 0.56)
- Conclusion: the "no detectable era difference in teammate gap" finding is not an artifact of longer modern seasons. It holds.

### Check C: Does the pit stop result depend on season length?
**The worry**
- Same issue: raw constructor points climb with more races, which could make the pit stop link look stronger than it is.

**What I did**
- Normalized constructor points to points-per-race, then re-ran the correlation with pit stop duration.

**What I found**
- The correlation barely moved (from -0.46 to -0.47)
- Conclusion: the pit-stop-to-points link is not inflated by longer modern calendars. It holds.

### What these checks told me overall
- Two of my three named limitations (season length in Tests 3 and 4) turned out not to change anything
- The one that mattered (DNFs in Test 2) I have now pinned to an exact effect: it explains some early-era movement but not the whole difference
- Listing a limitation is a hunch. Testing it is a finding. That is the difference here.

In [10]:
# sanity checks: testing the confounds I named in Tests 2, 3, and 4
# Instead of just flagging these limitations, I test whether they actually change my findings.

# --- Check A: is the Test 2 movement difference just caused by DNFs? ---
# If early-era drivers gained more places because more cars ahead RETIRED (not overtaking),
# then removing high-DNF races should shrink the early-vs-recent gap.
results['is_dnf'] = (results['statusId'] != 1).astype(int)
dnf_per_race = results.groupby('raceId')['is_dnf'].sum().rename('race_dnf_count')
res = results.merge(dnf_per_race, on='raceId')

rc = res[(res['position'].notna()) & (res['grid'] != 0)].copy()
rc['position_change'] = rc['grid'] - rc['position']

# keep only races with a below-median number of DNFs (the "clean" races)
thresh = dnf_per_race.median()
low_dnf = rc[rc['race_dnf_count'] <= thresh]

early_all = rc[rc['era'].isin(['2010-2013','2014-2020'])]['position_change']
recent_all = rc[rc['era'].isin(['2021','2022-2024'])]['position_change']
early_low = low_dnf[low_dnf['era'].isin(['2010-2013','2014-2020'])]['position_change']
recent_low = low_dnf[low_dnf['era'].isin(['2021','2022-2024'])]['position_change']

t1, p_all = stats.ttest_ind(early_all, recent_all, equal_var=False)
t2, p_low = stats.ttest_ind(early_low, recent_low, equal_var=False)
print('Check A: DNF confound on Test 2')
print('  ALL races     early:', round(early_all.mean(),2), 'recent:', round(recent_all.mean(),2), 'p:', round(p_all,5))
print('  LOW-DNF races early:', round(early_low.mean(),2), 'recent:', round(recent_low.mean(),2), 'p:', round(p_low,5))
print()

# --- Check B: does the Test 3 teammate gap picture change if I use points-PER-RACE? ---
# Confound: recent seasons have more races, inflating raw total-points gaps.
rs = results[~results['positionText'].isin(['W','F','E'])].copy()
tds = rs.groupby(['constructorId','driverId','year']).agg(
    season_points=('points','sum'), races=('raceId','nunique')).reset_index()
tds['ppr'] = tds['season_points'] / tds['races']
top2 = tds.sort_values(['races','season_points'], ascending=False).groupby(['constructorId','year']).head(2)
gap_ppr = top2.groupby(['constructorId','year']).agg(
    ppr_high=('ppr','max'), ppr_low=('ppr','min')).reset_index()
gap_ppr['gap_ppr'] = gap_ppr['ppr_high'] - gap_ppr['ppr_low']
gap_ppr['era'] = pd.cut(gap_ppr['year'], bins=[2009,2013,2020,2021,2024],
                        labels=['2010-2013','2014-2020','2021','2022-2024'])
groups = [gap_ppr[gap_ppr['era']==e]['gap_ppr'] for e in eras]
h, p_gap = stats.kruskal(*groups)
print('Check B: points-per-race teammate gap, omnibus p:', round(p_gap,4))
print()

# --- Check C: does the Test 4 pit stop link survive normalizing points by races? ---
races_per_year = results.groupby('year')['raceId'].nunique().rename('n_races')
ts = team_season.merge(races_per_year, on='year')
ts['constructor_points'] = ts['points_high'] + ts['points_low']
ts['points_per_race'] = ts['constructor_points'] / ts['n_races']
m = ts.merge(avg_pit_stop, on=['constructorId','year'], how='inner')
r_norm, p_norm = stats.pearsonr(m['avg_pit_stop_duration'], m['points_per_race'])
print('Check C: pit stop vs points-per-race, Pearson r:', round(r_norm,3), 'p:', round(p_norm,8))

Check A: DNF confound on Test 2
  ALL races     early: 1.44 recent: 0.83 p: 0.0
  LOW-DNF races early: 1.39 recent: 0.88 p: 0.00097

Check B: points-per-race teammate gap, omnibus p: 0.5622

Check C: pit stop vs points-per-race, Pearson r: -0.471 p: 0.0
